# Test shop only: LightGBM and Random Forest with time patterns

This notebook starts from the raw `test-shop/history.parquet` file. It does not read any prepared data or Sets A–D. It adds features for the average units sold in the same horizon week in prior years. Each stage displays a dataframe and writes a CSV snapshot to `Codex/outputs_time_pattern/` so the transformations are easy to inspect.

The task is to forecast the next 13 weeks for each of 2,904 articles. We compare LightGBM and Random Forest using rolling 13-week holdouts and the competition's per-article MASE score.

In [1]:
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'Codex' else Path('data/hackathon-retail-sets/Codex')
DATA_DIR = NOTEBOOK_DIR.parent
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs_time_pattern'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HORIZON = 13
print('Notebook folder:', NOTEBOOK_DIR.resolve())
print('Raw test data:', (DATA_DIR / 'test-shop' / 'history.parquet').resolve())
print('Snapshots go to:', OUTPUT_DIR.resolve())

Notebook folder: C:\Users\mehme\Desktop\Projects\District19\District19\data\hackathon-retail-sets\Codex
Raw test data: C:\Users\mehme\Desktop\Projects\District19\District19\data\hackathon-retail-sets\test-shop\history.parquet
Snapshots go to: C:\Users\mehme\Desktop\Projects\District19\District19\data\hackathon-retail-sets\Codex\outputs_time_pattern


## 1. Load the raw test-shop dataset

In [2]:
raw = pd.read_parquet(DATA_DIR / 'test-shop' / 'history.parquet')
raw['week_start'] = pd.to_datetime(raw['week_start'])
print('Shape:', raw.shape)
print('Types:\n', raw.dtypes)
display(raw.head())
raw.to_csv(OUTPUT_DIR / '01_raw_head.csv', index=False)

Shape: (415272, 4)
Types:
 article_id              str
week_start    datetime64[s]
units               float64
on_hand             float64
dtype: object


,article_id,week_start,units,on_hand
0,A100181,2020-07-27,5.0,5.0
1,A100181,2020-08-03,1.0,4.0
2,A100181,2020-08-10,1.0,15.0
3,A100181,2020-08-17,0.0,15.0
4,A100181,2020-08-24,1.0,14.0


## 2. Check keys and dates, then pivot to one row per article

In [3]:
print('Duplicate article-week rows:', raw.duplicated(['article_id', 'week_start']).sum())
print('Articles:', raw['article_id'].nunique())
print('Weeks:', raw['week_start'].nunique(), '| from', raw.week_start.min().date(), 'to', raw.week_start.max().date())

units = raw.pivot(index='article_id', columns='week_start', values='units').sort_index(axis=1)
units.columns.name = 'week_start'
print('Wide shape (articles x weeks):', units.shape)
print('Missing values:', int(units.isna().sum().sum()))
display(units.iloc[:5, :8])
units.reset_index().to_csv(OUTPUT_DIR / '02_wide_units.csv', index=False)

Duplicate article-week rows: 0
Articles: 2904
Weeks: 143 | from 2020-07-27 to 2023-04-17
Wide shape (articles x weeks): (2904, 143)
Missing values: 0


week_start,2020-07-27,2020-08-03,2020-08-10,2020-08-17,2020-08-24,2020-08-31,2020-09-07,2020-09-14
article_id,,,,,,,,
A100181,5.0,1.0,1.0,0.0,1.0,2.0,0.0,0.0
A100303,7.0,7.0,7.0,4.0,5.0,15.0,7.0,5.0
A100507,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A101368,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A101554,305.0,331.0,276.0,230.0,435.0,330.0,296.0,224.0


The dataset is expected to be a complete grid. The next cell checks that assumption and saves a compact summary of the series.

In [4]:
assert units.shape == (2904, 143)
assert units.notna().all().all()
assert (units.to_numpy() >= 0).all()

series_summary = pd.DataFrame({
    'article_id': units.index,
    'total_units': units.sum(axis=1).to_numpy(),
    'nonzero_weeks': (units > 0).sum(axis=1).to_numpy(),
    'zero_share': (units == 0).mean(axis=1).to_numpy(),
    'last_sale_week': [units.columns[np.flatnonzero(row > 0)[-1]] if (row > 0).any() else pd.NaT for row in units.to_numpy()]
})
display(series_summary.head())
display(series_summary[['total_units', 'nonzero_weeks', 'zero_share']].describe().T)
series_summary.to_csv(OUTPUT_DIR / '03_series_summary.csv', index=False)

,article_id,total_units,nonzero_weeks,zero_share,last_sale_week
0,A100181,168.0,99,0.307692,2023-04-17
1,A100303,876.0,142,0.006993,2023-04-17
2,A100507,260.0,14,0.902098,2023-04-10
3,A101368,680.0,19,0.867133,2023-04-10
4,A101554,31429.0,143,0.000000,2023-04-17


,count,mean,std,min,25%,50%,75%,max
total_units,2904.0,1476.394387,5910.313917,1.0,16.000000,92.000000,466.500000,120462.000000
nonzero_weeks,2904.0,50.538567,47.441106,1.0,8.000000,34.000000,89.000000,143.000000
zero_share,2904.0,0.646583,0.331756,0.0,0.377622,0.762238,0.944056,0.993007


## 3. Make simple history features

Each training example uses only the history available at its forecast origin. It contains recent lags, rolling summaries, sale frequency, and the requested horizon. The target is the actual units at that horizon.

In [5]:
Y = units.to_numpy(dtype=np.float32)
FEATURE_NAMES = (
    [f'lag_{k}' for k in [1, 2, 3, 4, 8, 13, 26, 52]]
    + [f'mean_{k}' for k in [4, 13, 26, 52]]
    + [f'median_{k}' for k in [4, 13, 26, 52]]
    + ['sale_rate_13', 'sale_rate_26', 'sale_rate_52', 'weeks_since_sale', 'horizon']
    + ['same_week_last_year', 'same_week_average', 'same_week_years_available']
)

def make_features(history, horizon):
    features = []
    for lag in [1, 2, 3, 4, 8, 13, 26, 52]:
        features.append(history[:, -lag] if history.shape[1] >= lag else np.zeros(len(history)))
    for window in [4, 13, 26, 52]:
        recent = history[:, -min(window, history.shape[1]):]
        features.append(recent.mean(axis=1))
    for window in [4, 13, 26, 52]:
        recent = history[:, -min(window, history.shape[1]):]
        features.append(np.median(recent, axis=1))
    for window in [13, 26, 52]:
        recent = history[:, -min(window, history.shape[1]):]
        features.append((recent > 0).mean(axis=1))
    sold = history > 0
    age = sold[:, ::-1].argmax(axis=1).astype(float)
    age[~sold.any(axis=1)] = history.shape[1]
    features.append(age)
    features.append(np.full(len(history), horizon))

    # Match the target horizon to the same week 52 and 104 weeks earlier.
    # These indices are always at or before the current history end.
    same_week_values = []
    for years_back in [1, 2]:
        index = history.shape[1] + horizon - 1 - 52 * years_back
        if index >= 0:
            same_week_values.append(history[:, index])
    if same_week_values:
        same_week = np.column_stack(same_week_values)
        features.append(same_week[:, 0])
        features.append(same_week.mean(axis=1))
        features.append(np.full(len(history), same_week.shape[1]))
    else:
        features.extend([np.zeros(len(history)) for _ in range(3)])
    return np.column_stack(features).astype(np.float32)

example = pd.DataFrame(make_features(Y[:, :100], 1), columns=FEATURE_NAMES)
display(example.head())
example.to_csv(OUTPUT_DIR / '04_feature_example.csv', index=False)

,lag_1,lag_2,lag_3,lag_4,lag_8,lag_13,lag_26,lag_52,mean_4,mean_13,mean_26,mean_52,median_4,median_13,median_26,median_52,sale_rate_13,sale_rate_26,sale_rate_52,weeks_since_sale,horizon,same_week_last_year,same_week_average,same_week_years_available
0,1.0,0.0,4.0,3.0,1.0,2.0,0.0,0.0,2.00,1.153846,1.000000,1.134615,2.0,1.0,1.0,1.0,0.692308,0.653846,0.673077,0.0,1.0,0.0,0.0,1.0
1,2.0,7.0,8.0,6.0,6.0,3.0,7.0,8.0,5.75,4.307693,5.153846,5.884615,6.5,4.0,5.5,6.0,1.000000,1.000000,1.000000,0.0,1.0,8.0,8.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,100.0,1.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,100.0,1.0,0.0,0.0,1.0
4,250.0,230.0,160.0,190.0,255.0,154.0,295.0,230.0,207.50,176.846161,213.961533,232.326920,210.0,165.0,220.0,226.0,1.000000,1.000000,1.000000,0.0,1.0,230.0,230.0,1.0


## 4. Train on rolling historical origins

We create examples every 4 weeks to keep the training table manageable. A training label is included only when its target week is before the validation cutoff, preventing future leakage.

In [6]:
def make_training_table(cutoff, origin_stride=4, min_history=52):
    rows, targets = [], []
    last_origin = cutoff - HORIZON
    for origin in range(min_history - 1, last_origin + 1, origin_stride):
        history = Y[:, :origin + 1]
        for h in range(1, HORIZON + 1):
            target_week = origin + h
            if target_week > cutoff:
                continue
            rows.append(make_features(history, h))
            targets.append(Y[:, target_week])
    X_train = np.vstack(rows)
    y_train = np.concatenate(targets)
    table = pd.DataFrame(X_train, columns=FEATURE_NAMES)
    table['target_units'] = y_train
    return table

# Preview the most recent validation split's training rows.
HOLDOUT_WEEKS = HORIZON
last_cutoff = Y.shape[1] - HOLDOUT_WEEKS - 1
training_preview = make_training_table(last_cutoff)
print('Training shape:', training_preview.shape)
display(training_preview.head())
training_preview.head(5000).to_csv(OUTPUT_DIR / '05_training_preview_first_5000.csv', index=False)

Training shape: (641784, 25)


,lag_1,lag_2,lag_3,lag_4,lag_8,lag_13,lag_26,lag_52,mean_4,mean_13,mean_26,mean_52,median_4,median_13,median_26,median_52,sale_rate_13,sale_rate_26,sale_rate_52,weeks_since_sale,horizon,same_week_last_year,same_week_average,same_week_years_available,target_units
0,3.0,2.0,3.0,0.0,0.0,2.0,1.0,5.0,2.00,1.153846,1.307692,1.384615,2.5,1.0,1.0,1.0,0.615385,0.769231,0.750000,0.0,1.0,5.0,5.0,1.0,1.0
1,1.0,14.0,4.0,8.0,3.0,2.0,8.0,7.0,6.75,5.230769,5.846154,6.576923,6.0,4.0,4.5,6.0,0.923077,0.961538,0.980769,0.0,1.0,7.0,7.0,1.0,14.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,52.0,1.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,52.0,1.0,0.0,0.0,1.0,0.0
4,312.0,320.0,191.0,230.0,289.0,390.0,245.0,305.0,263.25,245.153839,249.153839,267.442322,271.0,230.0,235.0,260.0,1.000000,1.000000,1.000000,0.0,1.0,305.0,305.0,1.0,210.0


## 5. Define LightGBM and Random Forest

Both models use the same features and training examples. LightGBM uses a median (quantile) objective, while Random Forest uses absolute-error splits/leaf predictions.

In [7]:
def make_models():
    return {
        'LightGBM': lgb.LGBMRegressor(
            objective='quantile', alpha=0.5, n_estimators=250, learning_rate=0.04,
            num_leaves=31, min_child_samples=100, subsample=0.8,
            colsample_bytree=0.9, reg_lambda=2.0, random_state=19, n_jobs=-1, verbosity=-1
        ),
        'RandomForest': RandomForestRegressor(
            n_estimators=100, min_samples_leaf=20, max_features=0.8,
            criterion='absolute_error', n_jobs=-1, random_state=19
        ),
    }

print('Models:', list(make_models()))

Models: ['LightGBM', 'RandomForest']


## 6. Score rolling 13-week holdouts

The score is `(1 - MASE) × 100`; higher is better. Each article is weighted equally. We keep the last two rolling windows as a check and choose between models using the older windows' final-half score (weeks 8–13).

In [8]:
def mase_score(cutoff, prediction):
    actual = Y[:, cutoff + 1:cutoff + HORIZON + 1]
    history = Y[:, :cutoff + 1]
    denominator = np.abs(np.diff(history, axis=1)).mean(axis=1)
    error = np.abs(actual - prediction)
    with np.errstate(divide='ignore', invalid='ignore'):
        scaled_error = np.where(denominator[:, None] > 0, error / denominator[:, None], np.nan)
    return {
        'score_all': 100 * (1 - np.nanmean(scaled_error)),
        'score_w1_w7': 100 * (1 - np.nanmean(scaled_error[:, :7])),
        'score_w8_w13': 100 * (1 - np.nanmean(scaled_error[:, 7:])),
    }

def predict_horizon(model, cutoff):
    history = Y[:, :cutoff + 1]
    X_future = np.vstack([make_features(history, h) for h in range(1, HORIZON + 1)])
    prediction = model.predict(X_future).reshape(HORIZON, Y.shape[0]).T
    return np.clip(prediction, 0, None)

cutoffs = [Y.shape[1] - HORIZON - 1 - 13 * i for i in range(4, -1, -1)]
score_rows = []
for cutoff in cutoffs:
    train = make_training_table(cutoff)
    X_train = train[FEATURE_NAMES].to_numpy()
    y_train = train['target_units'].to_numpy()
    for model_name, model in make_models().items():
        model.fit(X_train, y_train)
        prediction = predict_horizon(model, cutoff)
        score_rows.append({'model': model_name, 'cutoff': units.columns[cutoff].date(), **mase_score(cutoff, prediction)})

scores = pd.DataFrame(score_rows)
display(scores)
scores.to_csv(OUTPUT_DIR / '06_rolling_scores.csv', index=False)
summary = scores.groupby('model')[['score_all', 'score_w1_w7', 'score_w8_w13']].mean().sort_values('score_w8_w13', ascending=False)
display(summary)
summary.to_csv(OUTPUT_DIR / '07_model_summary.csv')

,model,cutoff,score_all,score_w1_w7,score_w8_w13
0,LightGBM,2022-01-17,5.155884,4.283442,6.173734
1,RandomForest,2022-01-17,1.576055,0.978757,2.272903
2,LightGBM,2022-04-18,-14.954825,-11.357095,-19.152177
3,RandomForest,2022-04-18,-16.733214,-13.022038,-21.062920
4,LightGBM,2022-07-18,-12.426999,-7.147668,-18.586218
5,RandomForest,2022-07-18,-14.316300,-9.309462,-20.157612
6,LightGBM,2022-10-17,-28.499130,-35.713025,-20.082919
7,RandomForest,2022-10-17,-30.299535,-36.465994,-23.105332
8,LightGBM,2023-01-16,-4.699002,-4.131474,-5.361118
9,RandomForest,2023-01-16,-9.710813,-9.554730,-9.892909


,score_all,score_w1_w7,score_w8_w13
model,,,
LightGBM,-11.084814,-10.813164,-11.401740
RandomForest,-13.896761,-13.474693,-14.389174


## 7. Fit the selected model on the full history and save predictions

This makes a new submission file in `Codex/outputs/`. Inspect the model summary and rolling scores first; the validation score is not a promise about the hidden competition weeks.

In [9]:
best_model_name = summary.index[0]
print('Selected by mean rolling w8–w13 score:', best_model_name)

final_train = make_training_table(Y.shape[1] - 1)
final_model = make_models()[best_model_name]
final_model.fit(final_train[FEATURE_NAMES].to_numpy(), final_train['target_units'].to_numpy())
final_prediction = predict_horizon(final_model, Y.shape[1] - 1)

submission = pd.DataFrame(final_prediction, index=units.index, columns=[f'w{i}' for i in range(1, HORIZON + 1)])
submission = submission.reset_index()
display(submission.head())
submission.to_csv(OUTPUT_DIR / f'submission_{best_model_name.lower()}.csv', index=False)
print('Saved:', OUTPUT_DIR / f'submission_{best_model_name.lower()}.csv')
print('Submission shape:', submission.shape)

Selected by mean rolling w8–w13 score: LightGBM


,article_id,w1,w2,w3,w4,w5,w6,w7,w8,w9,w10,w11,w12,w13
0,A100181,0.948393,0.948393,0.948393,0.948393,0.948393,0.948393,0.948393,0.948393,0.948393,0.948393,0.948393,0.948393,0.948393
1,A100303,5.847273,5.847273,5.847273,5.847273,5.847273,5.847273,5.892419,5.865694,5.865694,5.865694,5.847273,5.892419,5.847273
2,A100507,1.594040,1.594040,1.594040,1.594040,1.594040,1.594040,1.594040,1.594040,1.594040,1.594040,1.594040,1.594040,1.594040
3,A101368,12.698502,12.698502,12.698502,12.698502,12.698502,12.698502,12.698502,12.698502,12.698502,12.698502,12.698502,12.698502,12.698502
4,A101554,131.295097,127.501344,127.501344,124.921672,124.921672,128.259435,125.332009,127.501344,127.501344,127.501344,127.501344,128.259435,124.921672


Saved: c:\Users\mehme\Desktop\Projects\District19\District19\data\hackathon-retail-sets\Codex\outputs_time_pattern\submission_lightgbm.csv
Submission shape: (2904, 14)
